In [1]:
import tensorflow as tf
from tqdm import tqdm_notebook

print(tf.config.list_physical_devices('GPU'))

C:\Users\PC\.conda\envs\recora\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# ============================================================
# 0) INSTALL
# ============================================================
# pip install -U LibRecommender tensorflow pandas numpy clickhouse-connect scikit-learn

# If you are in notebook / interactive env and hit TF graph reuse issues:
# import tensorflow as tf
# tf.compat.v1.reset_default_graph()


# ============================================================
# 1) IMPORTS
# ============================================================
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import clickhouse_connect
from sklearn.preprocessing import StandardScaler, MinMaxScaler, QuantileTransformer

from recora.data import DatasetFeat

import pandas as pd

from recora.data import DatasetFeat, split_by_ratio_chrono

TRAIN_DAYS = 90
RANDOM_SEED = 42
TOP_K = 20

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore")

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

warnings.filterwarnings("ignore")

In [6]:
# ============================================================
# 2) GLOBAL CONFIG
# ============================================================
CLICKHOUSE_CONFIG = {
	"host": os.getenv("CH_HOST", "172.30.36.13"),
	"port": int(os.getenv("CH_PORT", "8123")),
	"username": os.getenv("CH_USER", "default"),
	"password": os.getenv("CH_PASSWORD", "192837465AhurA!@#"),
	"database": os.getenv("CH_DATABASE", "KF"),
	"secure": os.getenv("CH_SECURE", "false").lower() == "true",
}

# ============================================================
# 3) CLICKHOUSE CLIENT
# ============================================================
client = clickhouse_connect.get_client(**CLICKHOUSE_CONFIG)

# ============================================================
# 4) SQL QUERIES
#    KEEPING YOUR SQL UNCHANGED
# ============================================================
ITEM_SQL = f"""
WITH item_base AS (SELECT pp.category_level2_id AS category_id,
                          pp.category_level1_id AS root_category_id
                   FROM KF.plane_products pp
                   WHERE pp.category_level1_activity_type_id IN (1, 29)
                     AND pp.category_level2_activity_type_id IN (1, 29)
                     AND pp.product_id <> 0
                   GROUP BY pp.category_level2_id, pp.category_level1_id),

     availability_price AS (SELECT pp.category_level2_id          AS category_id,
                                   avg(isph.availability)         AS avg_availability,
                                   stddevSamp(isph.availability)  AS std_availability,

                                   avg(isph.avg_price)            AS avg_price,
                                   stddevSamp(isph.avg_price)     AS std_price,
                                   min(isph.avg_price)            AS min_price,
                                   max(isph.avg_price)            AS max_price,
                                   quantile(0.25)(isph.avg_price) AS p25_price,
                                   quantile(0.5)(isph.avg_price)  AS p50_price, -- corrected: median
                                   quantile(0.75)(isph.avg_price) AS p75_price
                            FROM KF.in_stock_price_history isph
                                     INNER JOIN KF.plane_products pp ON isph.shop_product_id = pp.shop_product_id
                            WHERE isph.snapshot_date > today() - {TRAIN_DAYS}
                              AND pp.category_level1_activity_type_id IN (1, 29)
                              AND pp.category_level2_activity_type_id IN (1, 29)
                              AND pp.product_id <> 0
                            GROUP BY pp.category_level2_id),

     popularity_discount AS (SELECT pp.category_level2_id                                        AS category_id,

                                    count(*)                                                     AS total_popularity,
                                    countIf(oi.order_date > today() - 30)                        AS recent_popularity,
                                    recent_popularity / total_popularity                         AS popularity_trend,

                                    sum(oi.items_jet_discount + oi.brand_discount + oi.items_vendor_discount) /
                                    nullIf(sum(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount +
                                               oi.brand_discount + oi.items_vendor_discount), 0) AS discount_ratio,

                                    topK(1)(oi.order_hour)[1]                                    AS mode_ordering_hour
                             FROM KF.fact_order_items oi
                                      INNER JOIN KF.plane_products pp ON oi.shop_product_id = pp.shop_product_id
                             WHERE oi.is_gross = 1
                               AND oi.order_date > today() - {TRAIN_DAYS}
                               AND pp.category_level1_activity_type_id IN (1, 29)
                               AND pp.category_level2_activity_type_id IN (1, 29)
                               AND pp.product_id <> 0
                             GROUP BY pp.category_level2_id),

     product_seller_stats AS (SELECT pp.category_level2_id              AS category_id,
                                     countDistinct(pp.shop_product_id)  AS num_products,
                                     countDistinct(pp.merchant_shop_id) AS num_sellers
                              FROM KF.plane_products pp
                              WHERE pp.category_level1_activity_type_id IN (1, 29)
                                AND pp.category_level2_activity_type_id IN (1, 29)
                                AND pp.product_id <> 0
                              GROUP BY pp.category_level2_id),

     first_appearance AS (SELECT pp.category_level2_id   AS category_id,
                                 min(isph.snapshot_date) AS first_seen_date
                          FROM KF.in_stock_price_history isph
                                   INNER JOIN KF.plane_products pp ON isph.shop_product_id = pp.shop_product_id
                          WHERE pp.category_level1_activity_type_id IN (1, 29)
                            AND pp.category_level2_activity_type_id IN (1, 29)
                            AND pp.product_id <> 0
                          GROUP BY pp.category_level2_id)

/*----- main SELECT: include all features -----*/
SELECT toString(b.category_id)                      AS category_id,
       toString(b.root_category_id)                 AS root_category_id,

       -- availability & price distribution
       ap.avg_availability,
       ap.std_availability,
       ap.avg_price,
       ap.std_price,
       ap.min_price,
       ap.max_price,
       ap.p25_price,
       ap.p50_price,
       ap.p75_price,

       -- popularity & discount
       p.total_popularity,
       p.recent_popularity,
       p.popularity_trend,
       p.discount_ratio                             AS discount_ratio,
       p.mode_ordering_hour                         AS mode_ordering_hour,

       -- product/seller variety
       ps.num_products,
       ps.num_sellers,

       -- item age (days since first appearance)
       dateDiff('day', fa.first_seen_date, today()) AS item_age_days
FROM item_base b
         JOIN availability_price ap
              ON b.category_id = ap.category_id
         JOIN popularity_discount p
              ON b.category_id = p.category_id
         JOIN product_seller_stats ps
              ON b.category_id = ps.category_id
         JOIN first_appearance fa
              ON b.category_id = fa.category_id
"""

USER_SQL = f"""
/*----- CTEs for item features -----*/
WITH
-- per‑user basic stats ------------------------------------------------
user_base AS (SELECT oi.user_id                                                                           AS user_id,
                     nullIf(argMax(ua.polygon_id, ua.address_id), 0)                                      AS polygon_id,

                     dateDiff('day', max(oi.order_date), today())                                         AS recency,
                     dateDiff('day', min(oi.order_date), today())                                         AS length,
                     countDistinct(oi.order_id)                                                           AS frequency,
                     sum(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount)            AS monetary,

                     countDistinct(pp.brand_id)                                                           AS brand_diversity,
                     countDistinct(pp.category_level2_id)                                                 AS category_diversity,
                     countDistinct(pp.category_level1_id)                                                 AS root_category_diversity,

                     monetary / frequency                                                                 AS aov,
                     count(oi.item_id) / frequency                                                        AS lpo, -- lines per order (number of item rows)
                     sum(oi.quantity) / countDistinct(oi.order_id)                                        AS ipo, -- items per order (total quantity)

                     avg(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount)            AS avg_item_spending,
                     min(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount)            AS min_item_spending,
                     max(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount)            AS max_item_spending,
                     quantile(0.25)(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount) AS p25_item_spending,
                     quantile(0.5)(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount)  AS p50_item_spending,
                     quantile(0.75)(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount) AS p75_item_spending,

                     sum(oi.items_jet_discount + oi.brand_discount + oi.items_vendor_discount) /
                     nullIf(sum(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount +
                                oi.brand_discount + oi.items_vendor_discount), 0)                         AS discount_ratio,

                     avg(oi.order_hour)                                                                   AS avg_ordering_hour

              FROM KF.fact_order_items AS oi
                       JOIN KF.dim_merchant_shops AS ms ON oi.merchant_shop_id = ms.merchant_shop_id
                       JOIN KF.dim_user_addresses AS ua ON oi.address_id = ua.address_id
                       JOIN KF.plane_products AS pp ON oi.shop_product_id = pp.shop_product_id
              WHERE oi.is_gross = 1
                AND oi.order_date > today() - {TRAIN_DAYS}
                AND ms.activity_type_id IN (1, 29)
                AND pp.category_level1_activity_type_id IN (1, 29)
                AND pp.category_level2_activity_type_id IN (1, 29)
              GROUP BY oi.user_id)

SELECT ub.user_id           AS user_id,
       ub.polygon_id,
       ub.recency,
       ub.frequency,
       ub.monetary,
       ub.length,

       ub.brand_diversity,
       ub.category_diversity,
       ub.root_category_diversity,

       ub.avg_ordering_hour AS user_avg_ordering_hour,

       ub.ipo,
       ub.lpo,
       ub.aov,

       ub.avg_item_spending,
       ub.min_item_spending,
       ub.max_item_spending,
       ub.p25_item_spending,
       ub.p50_item_spending,
       ub.p75_item_spending,

       ub.discount_ratio    AS user_discount_ratio
FROM user_base AS ub
"""

INTERACTION_SQL = f"""
SELECT oi.order_date                                                       AS order_date,
       oi.order_hour                                                       AS order_hour,
       oi.user_id                                                          AS user_id,
       oi.items_jet_discount                                               AS jet_discount,
       oi.items_voucher_discount                                           AS voucher_discount,
       toString(pp.category_level2_id)                                     AS category_id
FROM KF.fact_order_items oi
         INNER JOIN KF.plane_products pp
                    ON oi.shop_product_id = pp.shop_product_id
WHERE oi.is_gross = 1
  AND oi.order_date > today() - {TRAIN_DAYS}
  AND pp.category_level1_activity_type_id IN (1, 29)
  AND pp.category_level2_activity_type_id IN (1, 29)
  AND pp.product_id != 0
"""
#
# # ============================================================
# # 5) READ DATA
# # ============================================================
item_df = client.query_df(ITEM_SQL)
user_df = client.query_df(USER_SQL)
inter_events = client.query_df(INTERACTION_SQL)
#
# print("item_df shape      :", item_df.shape)
# print("user_df shape      :", user_df.shape)
# print("inter_events shape :", inter_events.shape)

In [7]:
item_df.to_csv('./sample_data/recora_category_item_df.csv')
user_df.to_csv('./sample_data/recora_category_user_df.csv')
inter_events.to_csv('./sample_data/recora_category_inter_events.csv')

In [8]:
item_df = pd.read_csv('./sample_data/recora_category_item_df.csv', index_col=0).rename(columns={'category_id': 'item'})
user_df = pd.read_csv('./sample_data/recora_category_user_df.csv', index_col=0).rename(columns={'user_id': 'user'})
inter_events = pd.read_csv('./sample_data/recora_category_inter_events.csv', index_col=0).rename(
	columns={'category_id': 'item', 'user_id': 'user', 'order_date': 'time'})

In [9]:
inter_events['time'] = pd.to_datetime(inter_events['time']).astype('int64') // 10 ** 9
inter_events = inter_events.sort_values(by='time')

In [10]:
# inter_events = inter_events[(inter_events['jet_discount'] == 0) & ((inter_events['voucher_discount'] == 0))]

In [11]:
print("item_df features:\n", item_df.columns.tolist(), "\n")
print("user_df features:\n", user_df.columns.tolist(), "\n")
print("inter_events features:\n", inter_events.columns.tolist(), "\n")

item_df features:
 ['item', 'root_category_id', 'avg_availability', 'std_availability', 'avg_price', 'std_price', 'min_price', 'max_price', 'p25_price', 'p50_price', 'p75_price', 'total_popularity', 'recent_popularity', 'popularity_trend', 'discount_ratio', 'mode_ordering_hour', 'num_products', 'num_sellers', 'item_age_days'] 

user_df features:
 ['user', 'polygon_id', 'recency', 'frequency', 'monetary', 'length', 'brand_diversity', 'category_diversity', 'root_category_diversity', 'user_avg_ordering_hour', 'ipo', 'lpo', 'aov', 'avg_item_spending', 'min_item_spending', 'max_item_spending', 'p25_item_spending', 'p50_item_spending', 'p75_item_spending', 'user_discount_ratio'] 

inter_events features:
 ['time', 'order_hour', 'user', 'jet_discount', 'voucher_discount', 'item'] 



In [31]:
pareto = inter_events.groupby('item').size().rename("pareto").sort_values(ascending=False).cumsum()
pareto = pareto / pareto.max()
pareto = pareto[pareto <= 0.95].reset_index()
pareto

,item,pareto
0,33,0.068266
1,69,0.117080
2,3100,0.158353
3,37,0.197741
4,3070,0.236590
...,...,...
62,76,0.937366
63,3142,0.940512
64,22,0.943495
65,3139,0.946463


In [32]:
filtered_users = inter_events.groupby('user')['item'].count().reset_index()
filtered_users = filtered_users[filtered_users['item'] > 0]

inter_events = inter_events.loc[
	(inter_events['user'].isin(filtered_users['user'])) &
	(inter_events['item'].isin(pareto['item']))
	]

inter_events['user'] = inter_events['user'].astype(str)
inter_events['item'] = inter_events['item'].astype(str)

item_df['item'] = item_df['item'].astype(str)
user_df['user'] = user_df['user'].astype(str)

In [33]:
df = (
	inter_events.
	merge(user_df, how="inner", on="user", ).
	merge(item_df, how="inner", on="item", )
)

df['label'] = 1

# df.loc[df['label'] > 10, 'label'] = 10

# # 1 + 2 + ... + n = (n + 1) * n / 2
# df['label'] = (df['label'] * (df['label'] + 1)) / 2

In [34]:
df.shape

(5016842, 44)

In [35]:
# import pandas as pd
#
# # Count interactions per user and item
# item_counts = df['item'].value_counts()
#
# # Apply filtering thresholds
# df = df[
# 	(df['item'].map(item_counts) >= 90)
# ].copy()

In [39]:
user_df.columns

Index(['user', 'polygon_id', 'recency', 'frequency', 'monetary', 'length',
       'brand_diversity', 'category_diversity', 'root_category_diversity',
       'user_avg_ordering_hour', 'ipo', 'lpo', 'aov', 'avg_item_spending',
       'min_item_spending', 'max_item_spending', 'p25_item_spending',
       'p50_item_spending', 'p75_item_spending', 'user_discount_ratio'],
      dtype='object')

In [41]:
# Single sparse columns
single_sparse_feature_columns = [
	"root_category_id",
	"polygon_id"
]

# Multi sparse columns (list of lists for multi‑hot encoding)
multi_sparse_feature_columns = [
	[]
]

# All dense features (both user and item sides, plus order_hour)
dense_feature_columns = [
	# Item dense features
	'avg_availability', 'std_availability',
	'avg_price', 'std_price', 'min_price', 'max_price', 'p25_price',
	'p50_price', 'p75_price', 'total_popularity', 'recent_popularity',
	'popularity_trend', 'discount_ratio', 'mode_ordering_hour',
	'num_products', 'num_sellers', 'item_age_days',

	# User dense features
	"recency", "length", "frequency", "monetary",
	"brand_diversity", "category_diversity", "root_category_diversity",
	"user_avg_ordering_hour", "ipo", "lpo", "aov",
	"avg_item_spending", "min_item_spending", "max_item_spending",
	"p25_item_spending", "p50_item_spending", "p75_item_spending",
	"user_discount_ratio",

	# Interaction‑level feature
	"order_hour",
	# "time"
]

# Features belonging to the user side
user_feature_columns = [
	# User sparse
	"polygon_id",

	# User dense
	'recency', 'frequency', 'monetary', 'length',
	'brand_diversity', 'category_diversity', 'root_category_diversity',
	'user_avg_ordering_hour', 'ipo', 'lpo', 'aov', 'avg_item_spending',
	'min_item_spending', 'max_item_spending', 'p25_item_spending',
	'p50_item_spending', 'p75_item_spending', 'user_discount_ratio',
	# Interaction‑level (treated as user feature for simplicity)
	"order_hour",
]

# Features belonging to the item side
item_feature_columns = [
	# Item sparse
	"root_category_id",

	# Item dense
	'avg_availability', 'std_availability',
	'avg_price', 'std_price', 'min_price', 'max_price', 'p25_price',
	'p50_price', 'p75_price', 'total_popularity', 'recent_popularity',
	'popularity_trend', 'discount_ratio', 'mode_ordering_hour',
	'num_products', 'num_sellers', 'item_age_days',
]

df = df[['user', 'item', 'time', 'label'] + user_feature_columns + item_feature_columns]
df = df.fillna(0)

In [42]:
 # Handle single sparse columns
for c in single_sparse_feature_columns:
	for df in [df]:
		df[c] = df[c].astype(str).fillna("missing")
		df.loc[df[c] == "0", c] = "missing"

# # Handle multi sparse columns
# multi_sparse_flat = [col for group in multi_sparse_col for col in group]
# for c in multi_sparse_flat:
# 	for df in [df]:
# 		df[c] = df[c].astype(str).fillna("missing")
# 		df.loc[df[c] == "0", c] = "missing"

scaler = QuantileTransformer(output_distribution='uniform', random_state=42, n_quantiles=25)
df[dense_feature_columns] = scaler.fit_transform(df[dense_feature_columns]).astype(np.float32)

In [43]:
train_data, eval_data = split_by_ratio_chrono(df, order=True, test_size=0.2)

In [44]:
FREQ_RECENCY_SW = False


# Weight recent repeat interactions higher during training.
def add_frequency_recency_sample_weight(frame, user_col="user", item_col="item", time_col="time"):
	weighted = frame.copy()

	pair_frequency = weighted.groupby([user_col, item_col])[item_col].transform("size").astype(np.float32)
	user_first_time = weighted.groupby(user_col)[time_col].transform("min")
	user_last_time = weighted.groupby(user_col)[time_col].transform("max")
	user_time_span = (user_last_time - user_first_time).replace(0, 1).astype(np.float32)
	recency_score = (1.0 + (weighted[time_col] - user_first_time) / user_time_span).astype(np.float32)

	weighted["sample_weight"] = (pair_frequency * recency_score).astype(np.float32)
	return weighted


if FREQ_RECENCY_SW:
	train_data = add_frequency_recency_sample_weight(train_data)
else:
	train_data['sample_weight'] = 1

train_data[["user", "item", "time", "sample_weight"]].head()

,user,item,time,sample_weight
4572254,1,37,1774224000,1
4578464,1,3122,1774224000,1
4579052,1,98,1774224000,1
4579122,1,46,1774224000,1
4579571,1,3093,1774224000,1


In [45]:
train_set, data_info = DatasetFeat.build_trainset(
	train_data=train_data,
	user_col=user_feature_columns,
	item_col=item_feature_columns,
	sparse_col=single_sparse_feature_columns,
	dense_col=dense_feature_columns,
	# multi_sparse_col=multi_sparse_col,
	# pad_val=["missing", "missing"],
	unique_feat=False
)

# train_set, data_info = DatasetFeat.build_trainset(
# 	train_data=train_data,
# 	user_col=[c for c in user_col],
# 	# item_col=item_col,
# 	sparse_col=[c for c in single_sparse_col if c not in item_col],
# 	dense_col=[c for c in dense_col if c not in item_col],
# 	shuffle=False,
# 	# multi_sparse_col=[c for c in multi_sparse_col if c not in item_col],
# 	# pad_val=["missing", "missing"],
# 	# TODO: important
# 	unique_feat=False
# )

eval_set = DatasetFeat.build_testset(eval_data)

In [46]:
# from recora.data import DatasetPure

# train_set, data_info = DatasetPure.build_trainset(train_data)
# eval_set = DatasetPure.build_evalset(eval_data)

# # del train_data, eval_data

In [75]:
# # Transformers
# hp = {
# 	"NEG_SAMPLING": True,
# 	"NEG_SAMPLER": "unconsumed",  # random | unconsumed | popular
# 	"NUM_NEG": 1,
# 	"BATCH_SIZE": 4096,
# 	"LR": 1e-3,
# 	"LOSS_TYPE": "lambdarank",
# 	"TASK": "ranking",
# 	"EMBED_SIZE": 4,
# 	"N_EPOCHS": 5,
# 	"LR_DECAY": True,
# 	"RECENT_NUM": 15,
# 	"NUM_HEAD": 2,
# 	"NUM_TFM_LAYERS": 1
# }

# RNN4Rec
hp = {
	"NEG_SAMPLING": True,
	"NEG_SAMPLER": "unconsumed",  # random | unconsumed | popular
	"NUM_NEG": 1,
	"BATCH_SIZE": 4096,
	"LR": 1e-3,
	"LOSS_TYPE": "cross_entropy",
	"TASK": "ranking",
	"EMBED_SIZE": 32,
	"N_EPOCHS": 3,
	"LR_DECAY": True,
	"RECENT_NUM": 15,
	"HIDDEN_UNITS": (32, 32)
}

#  ============================== RNN4Rec ==============================
# Hyperparameters:
# ----------------------------------------
# {'NEG_SAMPLING': True, 'NEG_SAMPLER': 'unconsumed', 'NUM_NEG': 1, 'BATCH_SIZE': 4096, 'LR': 0.001, 'LOSS_TYPE': 'cross_entropy', 'TASK': 'ranking', 'EMBED_SIZE': 32, 'N_EPOCHS': 5, 'LR_DECAY': True, 'RECENT_NUM': 15, 'HIDDEN_UNITS': (32, 32)}
# ----------------------------------------
# Training start time: 2026-04-07 01:44:53
# total params: 163,653 | embedding params: 150,341 | network params: 13,312
# With lr_decay, epoch 1 learning rate: 0.0010000000474974513
# train: 100%|██████████| 1647/1647 [05:17<00:00,  5.19it/s]
#
# Epoch 1 elapsed: 317.444s
# 	 train_loss: 0.4262
# eval_pointwise: 100%|██████████| 411/411 [00:00<00:00, 497.54it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:01<00:00, 1273.05it/s]
# 	 eval log_loss: 0.4060
# 	 eval balanced_accuracy: 0.8215
# 	 eval roc_auc: 0.8973
# 	 eval pr_auc: 0.8959
# 	 eval precision@20: 0.0326
# 	 eval recall@20: 0.1308
# 	 eval map@20: 0.1243
# 	 eval ndcg@20: 0.2042
# ==============================
# With lr_decay, epoch 2 learning rate: 0.0009600000339560211
# train: 100%|██████████| 1647/1647 [05:12<00:00,  5.27it/s]
#
# Epoch 2 elapsed: 312.810s
# 	 train_loss: 0.4015
# eval_pointwise: 100%|██████████| 411/411 [00:00<00:00, 513.67it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:02<00:00, 1184.50it/s]
# 	 eval log_loss: 0.4015
# 	 eval balanced_accuracy: 0.8234
# 	 eval roc_auc: 0.9000
# 	 eval pr_auc: 0.8977
# 	 eval precision@20: 0.0323
# 	 eval recall@20: 0.1304
# 	 eval map@20: 0.1218
# 	 eval ndcg@20: 0.2012
# ==============================
# With lr_decay, epoch 3 learning rate: 0.0009215999743901193
# train: 100%|██████████| 1647/1647 [07:34<00:00,  3.62it/s]
#
# Epoch 3 elapsed: 454.842s
# 	 train_loss: 0.3926
# eval_pointwise: 100%|██████████| 411/411 [00:00<00:00, 506.89it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:02<00:00, 1117.11it/s]
# 	 eval log_loss: 0.3923
# 	 eval balanced_accuracy: 0.8284
# 	 eval roc_auc: 0.9048
# 	 eval pr_auc: 0.9030
# 	 eval precision@20: 0.0342
# 	 eval recall@20: 0.1394
# 	 eval map@20: 0.1314
# 	 eval ndcg@20: 0.2137
# ==============================
# With lr_decay, epoch 4 learning rate: 0.0008847358985804021
# train:  27%|██▋       | 451/1647 [01:44<04:35,  4.34it/s]

# YoutubeRanker
hp = {
	"NEG_SAMPLING": True,
	"NEG_SAMPLER": "unconsumed",  # random | unconsumed | popular
	"NUM_NEG": 3,
	"BATCH_SIZE": 4096,
	"LR": 1e-3,
	"LOSS_TYPE": "ranknet",
	"TASK": "ranking",
	"EMBED_SIZE": 8,
	"N_EPOCHS": 3,
	"LR_DECAY": True,
	"RECENT_NUM": 31,
	"RANDOM_NUM": 31,
	"HIDDEN_UNITS": (256, 128, 64),
}

#  ============================== YouTubeRanking ==============================
# Hyperparameters:
# ----------------------------------------
# {'NEG_SAMPLING': True, 'NEG_SAMPLER': 'unconsumed', 'NUM_NEG': 1, 'BATCH_SIZE': 4096, 'LR': 0.001, 'LOSS_TYPE': 'ranknet', 'TASK': 'ranking', 'EMBED_SIZE': 32, 'N_EPOCHS': 3, 'LR_DECAY': True, 'RECENT_NUM': 31, 'RANDOM_NUM': 31, 'HIDDEN_UNITS': (256, 128, 64)}
# ----------------------------------------
# Training start time: 2026-04-07 02:17:46
# total params: 4,500,769 | embedding params: 4,033,761 | network params: 467,008
# With lr_decay, epoch 1 learning rate: 0.0010000000474974513
# train: 100%|██████████| 824/824 [01:37<00:00,  8.42it/s]
#
# Epoch 1 elapsed: 97.888s
# 	 train_loss: 0.2224
# eval_pointwise: 100%|██████████| 411/411 [00:05<00:00, 81.19it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:24<00:00, 103.02it/s]
#
# 	 eval log_loss: 0.7741
# 	 eval balanced_accuracy: 0.6796
# 	 eval roc_auc: 0.9088
# 	 eval pr_auc: 0.9073
# 	 eval precision@20: 0.0361
# 	 eval recall@20: 0.1495
# 	 eval map@20: 0.1411
# 	 eval ndcg@20: 0.2279
# ==============================
# With lr_decay, epoch 2 learning rate: 0.0009600000339560211
# train: 100%|██████████| 824/824 [01:32<00:00,  8.92it/s]
#
# Epoch 2 elapsed: 92.424s
# 	 train_loss: 0.1976
# eval_pointwise: 100%|██████████| 411/411 [00:06<00:00, 65.19it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:30<00:00, 80.65it/s]
#
# 	 eval log_loss: 0.6700
# 	 eval balanced_accuracy: 0.7221
# 	 eval roc_auc: 0.9104
# 	 eval pr_auc: 0.9087
# 	 eval precision@20: 0.0376
# 	 eval recall@20: 0.1547
# 	 eval map@20: 0.1470
# 	 eval ndcg@20: 0.2355
# ==============================
# With lr_decay, epoch 3 learning rate: 0.0009215999743901193
# train: 100%|██████████| 824/824 [01:27<00:00,  9.43it/s]
#
# Epoch 3 elapsed: 87.389s
# 	 train_loss: 0.1884
# eval_pointwise: 100%|██████████| 411/411 [00:05<00:00, 74.51it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:31<00:00, 79.62it/s]
#
# 	 eval log_loss: 0.5902
# 	 eval balanced_accuracy: 0.7372
# 	 eval roc_auc: 0.9038
# 	 eval pr_auc: 0.8924
# 	 eval precision@20: 0.0386
# 	 eval recall@20: 0.1610
# 	 eval map@20: 0.1502
# 	 eval ndcg@20: 0.2411
# ==============================

#  ============================== YouTubeRanking ==============================
# Hyperparameters:
# ----------------------------------------
# {'NEG_SAMPLING': True, 'NEG_SAMPLER': 'unconsumed', 'NUM_NEG': 1, 'BATCH_SIZE': 4096, 'LR': 0.001, 'LOSS_TYPE': 'ranknet', 'TASK': 'ranking', 'EMBED_SIZE': 32, 'N_EPOCHS': 3, 'LR_DECAY': True, 'RECENT_NUM': 15, 'HIDDEN_UNITS': (256, 128, 64)}
# ----------------------------------------
# Training start time: 2026-04-07 02:09:11
# total params: 4,500,769 | embedding params: 4,033,761 | network params: 467,008
# With lr_decay, epoch 1 learning rate: 0.0010000000474974513
# train: 100%|██████████| 824/824 [01:29<00:00,  9.22it/s]
#
# Epoch 1 elapsed: 89.344s
# 	 train_loss: 0.2223
# eval_pointwise: 100%|██████████| 411/411 [00:07<00:00, 55.50it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:21<00:00, 116.08it/s]
#
# 	 eval log_loss: 0.7879
# 	 eval balanced_accuracy: 0.6739
# 	 eval roc_auc: 0.9019
# 	 eval pr_auc: 0.8973
# 	 eval precision@20: 0.0366
# 	 eval recall@20: 0.1519
# 	 eval map@20: 0.1378
# 	 eval ndcg@20: 0.2260
# ==============================
# With lr_decay, epoch 2 learning rate: 0.0009600000339560211
# train: 100%|██████████| 824/824 [01:22<00:00,  9.97it/s]
#
# Epoch 2 elapsed: 82.691s
# 	 train_loss: 0.1978
# eval_pointwise: 100%|██████████| 411/411 [00:05<00:00, 71.81it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:32<00:00, 78.12it/s]
#
# 	 eval log_loss: 0.7506
# 	 eval balanced_accuracy: 0.6954
# 	 eval roc_auc: 0.9085
# 	 eval pr_auc: 0.9057
# 	 eval precision@20: 0.0377
# 	 eval recall@20: 0.1554
# 	 eval map@20: 0.1470
# 	 eval ndcg@20: 0.2362
# ==============================
# With lr_decay, epoch 3 learning rate: 0.0009215999743901193
# train: 100%|██████████| 824/824 [01:38<00:00,  8.37it/s]
#
# Epoch 3 elapsed: 98.491s
# 	 train_loss: 0.1887
# eval_pointwise: 100%|██████████| 411/411 [00:05<00:00, 72.72it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:28<00:00, 89.11it/s]
#
# 	 eval log_loss: 0.7259
# 	 eval balanced_accuracy: 0.6876
# 	 eval roc_auc: 0.9024
# 	 eval pr_auc: 0.8928
# 	 eval precision@20: 0.0379
# 	 eval recall@20: 0.1581
# 	 eval map@20: 0.1485
# 	 eval ndcg@20: 0.2387
# ==============================

# # BPR
# hp = {
# 	"NEG_SAMPLING": True,
# 	"NEG_SAMPLER": "unconsumed",  # random | unconsumed | popular
# 	"NUM_NEG": 5,
# 	"BATCH_SIZE": 4096,
# 	"LR": 1e-3,
# 	"LOSS_TYPE": "bpr",
# 	"TASK": "ranking",
# 	"EMBED_SIZE": 8,
# 	"N_EPOCHS": 3,
# 	"LR_DECAY": True,
# }

#  ============================== BPR ==============================
# Hyperparameters:
# ----------------------------------------
# {'NEG_SAMPLING': True, 'NEG_SAMPLER': 'unconsumed', 'NUM_NEG': 31, 'BATCH_SIZE': 4096, 'LR': 0.001, 'LOSS_TYPE': 'bpr', 'TASK': 'ranking', 'EMBED_SIZE': 32, 'N_EPOCHS': 3, 'LR_DECAY': True}
# ----------------------------------------
# Training start time: 2026-04-07 03:05:26
# With lr_decay, epoch 1 learning rate: 0.0010000000474974513
# train: 100%|██████████| 25545/25545 [04:07<00:00, 103.40it/s]
#
# Epoch 1 elapsed: 247.055s
# 	 train_loss: 0.278
# eval_pointwise: 100%|██████████| 411/411 [00:00<00:00, 522.92it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:01<00:00, 1328.12it/s]
#
# 	 eval log_loss: 0.3993
# 	 eval balanced_accuracy: 0.8254
# 	 eval roc_auc: 0.9024
# 	 eval pr_auc: 0.9020
# 	 eval precision@20: 0.0335
# 	 eval recall@20: 0.1346
# 	 eval map@20: 0.1222
# 	 eval ndcg@20: 0.2054
# ==============================
# With lr_decay, epoch 2 learning rate: 0.0009600000339560211
# train: 100%|██████████| 25545/25545 [03:59<00:00, 106.72it/s]
#
# Epoch 2 elapsed: 239.381s
# 	 train_loss: 0.2023
# eval_pointwise: 100%|██████████| 411/411 [00:00<00:00, 508.64it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:01<00:00, 1271.84it/s]
#
# 	 eval log_loss: 0.3847
# 	 eval balanced_accuracy: 0.8339
# 	 eval roc_auc: 0.9090
# 	 eval pr_auc: 0.9086
# 	 eval precision@20: 0.0354
# 	 eval recall@20: 0.1475
# 	 eval map@20: 0.1369
# 	 eval ndcg@20: 0.2213
# ==============================
# With lr_decay, epoch 3 learning rate: 0.0009215999743901193
# train: 100%|██████████| 25545/25545 [04:00<00:00, 106.22it/s]
#
# Epoch 3 elapsed: 240.495s
# 	 train_loss: 0.1618
# eval_pointwise: 100%|██████████| 411/411 [00:00<00:00, 506.16it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:02<00:00, 1228.23it/s]
#
# 	 eval log_loss: 0.4205
# 	 eval balanced_accuracy: 0.8181
# 	 eval roc_auc: 0.9048
# 	 eval pr_auc: 0.9050
# 	 eval precision@20: 0.0353
# 	 eval recall@20: 0.1491
# 	 eval map@20: 0.1398
# 	 eval ndcg@20: 0.2247
# ==============================

#  ============================== BPR ==============================
# Hyperparameters:
# ----------------------------------------
# {'NEG_SAMPLING': True, 'NEG_SAMPLER': 'unconsumed', 'NUM_NEG': 5, 'BATCH_SIZE': 4096, 'LR': 0.001, 'LOSS_TYPE': 'bpr', 'TASK': 'ranking', 'EMBED_SIZE': 8, 'N_EPOCHS': 3, 'LR_DECAY': True}
# ----------------------------------------
# Training start time: 2026-04-07 03:28:38
# With lr_decay, epoch 1 learning rate: 0.0010000000474974513
# train: 100%|██████████| 4118/4118 [00:47<00:00, 87.37it/s]
#
# Epoch 1 elapsed: 47.131s
# 	 train_loss: 0.3696
# eval_pointwise: 100%|██████████| 411/411 [00:00<00:00, 502.51it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:02<00:00, 886.44it/s]
#
# 	 eval log_loss: 0.4330
# 	 eval balanced_accuracy: 0.8139
# 	 eval roc_auc: 0.8907
# 	 eval pr_auc: 0.8892
# 	 eval precision@20: 0.0304
# 	 eval recall@20: 0.1202
# 	 eval map@20: 0.1141
# 	 eval ndcg@20: 0.1905
# ==============================
# With lr_decay, epoch 2 learning rate: 0.0009600000339560211
# train: 100%|██████████| 4118/4118 [00:47<00:00, 86.03it/s]
#
# Epoch 2 elapsed: 47.875s
# 	 train_loss: 0.264
# eval_pointwise: 100%|██████████| 411/411 [00:01<00:00, 349.65it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:02<00:00, 1170.80it/s]
#
# 	 eval log_loss: 0.4263
# 	 eval balanced_accuracy: 0.8129
# 	 eval roc_auc: 0.8931
# 	 eval pr_auc: 0.8912
# 	 eval precision@20: 0.0305
# 	 eval recall@20: 0.1205
# 	 eval map@20: 0.1155
# 	 eval ndcg@20: 0.1917
# ==============================
# With lr_decay, epoch 3 learning rate: 0.0009215999743901193
# train: 100%|██████████| 4118/4118 [00:57<00:00, 71.33it/s]
#
# Epoch 3 elapsed: 57.738s
# 	 train_loss: 0.2559
# eval_pointwise: 100%|██████████| 411/411 [00:00<00:00, 448.43it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:01<00:00, 1451.24it/s]
#
# 	 eval log_loss: 0.4254
# 	 eval balanced_accuracy: 0.8122
# 	 eval roc_auc: 0.8945
# 	 eval pr_auc: 0.8923
# 	 eval precision@20: 0.0307
# 	 eval recall@20: 0.1223
# 	 eval map@20: 0.1171
# 	 eval ndcg@20: 0.1940
# ==============================

# # DeepFM
# hp = {
# 	"NEG_SAMPLING": True,
# 	"NEG_SAMPLER": "unconsumed",  # random | unconsumed | popular
# 	"NUM_NEG": 5,
# 	"BATCH_SIZE": 4096,
# 	"LR": 1e-4,
# 	"LOSS_TYPE": "ranknet",
# 	"TASK": "ranking",
# 	"EMBED_SIZE": 8,
# 	"N_EPOCHS": 3,
# 	"LR_DECAY": True,
# 	"HIDDEN_UNITS": (256, 128, 64),
# }

#  ============================== DeepFM ==============================
# Hyperparameters:
# ----------------------------------------
# {'NEG_SAMPLING': True, 'NEG_SAMPLER': 'unconsumed', 'NUM_NEG': 5, 'BATCH_SIZE': 4096, 'LR': 0.0001, 'LOSS_TYPE': 'ranknet', 'TASK': 'ranking', 'EMBED_SIZE': 8, 'N_EPOCHS': 3, 'LR_DECAY': True, 'HIDDEN_UNITS': (256, 128, 64)}
# ----------------------------------------
# Training start time: 2026-04-07 03:53:05
# total params: 1,280,351 | embedding params: 1,134,819 | network params: 145,532
# With lr_decay, epoch 1 learning rate: 9.999999747378752e-05
# train: 100%|██████████| 4118/4118 [02:58<00:00, 23.12it/s]
#
# Epoch 1 elapsed: 178.118s
# 	 train_loss: 0.2453
# eval_pointwise: 100%|██████████| 411/411 [00:04<00:00, 90.15it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:21<00:00, 113.91it/s]
#
# 	 eval log_loss: 0.5230
# 	 eval balanced_accuracy: 0.7689
# 	 eval roc_auc: 0.8788
# 	 eval pr_auc: 0.8801
# 	 eval precision@20: 0.0342
# 	 eval recall@20: 0.1391
# 	 eval map@20: 0.1310
# 	 eval ndcg@20: 0.2144
# ==============================
# With lr_decay, epoch 2 learning rate: 9.599999611964449e-05
# train: 100%|██████████| 4118/4118 [03:03<00:00, 22.46it/s]
#
# Epoch 2 elapsed: 183.378s
# 	 train_loss: 0.2268
# eval_pointwise: 100%|██████████| 411/411 [00:04<00:00, 90.27it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:20<00:00, 119.59it/s]
#
# 	 eval log_loss: 0.5190
# 	 eval balanced_accuracy: 0.7729
# 	 eval roc_auc: 0.8855
# 	 eval pr_auc: 0.8859
# 	 eval precision@20: 0.0356
# 	 eval recall@20: 0.1463
# 	 eval map@20: 0.1315
# 	 eval ndcg@20: 0.2186
# ==============================
# With lr_decay, epoch 3 learning rate: 9.215999307343736e-05
# train: 100%|██████████| 4118/4118 [03:01<00:00, 22.71it/s]
#
# Epoch 3 elapsed: 181.361s
# 	 train_loss: 0.2102
# eval_pointwise: 100%|██████████| 411/411 [00:04<00:00, 92.07it/s]
# eval_listwise: 100%|██████████| 2500/2500 [00:20<00:00, 119.25it/s]
#
# 	 eval log_loss: 0.5002
# 	 eval balanced_accuracy: 0.7823
# 	 eval roc_auc: 0.8945
# 	 eval pr_auc: 0.8928
# 	 eval precision@20: 0.0358
# 	 eval recall@20: 0.1477
# 	 eval map@20: 0.1384
# 	 eval ndcg@20: 0.2251
# ==============================
#
# # LightGCN
# hp = {
# 	"NEG_SAMPLING": True,
# 	"NEG_SAMPLER": "unconsumed",  # random | unconsumed | popular
# 	"NUM_NEG": 5,
# 	"N_LAYERS": 3,
# 	"BATCH_SIZE": 4096,
# 	"LR": 1e-3,
# 	"LOSS_TYPE": "bpr",
# 	"TASK": "ranking",
# 	"EMBED_SIZE": 16,
# 	"N_EPOCHS": 3,
# 	"LR_DECAY": True,
# }

# # NGCF
# hp = {
# 	"NEG_SAMPLING": True,
# 	"NEG_SAMPLER": "unconsumed",  # random | unconsumed | popular
# 	"NUM_NEG": 5,
# 	"N_LAYERS": 3,
# 	"BATCH_SIZE": 4096,
# 	"LR": 1e-4,
# 	"LOSS_TYPE": "bpr",
# 	"TASK": "ranking",
# 	"EMBED_SIZE": 16,
# 	"LAYER_SIZES": (16, 16, 16),
# 	"NODE_DROPOUT": 0.1,
# 	"MESSAGE_DROPOUT": 0.1,
# 	"N_EPOCHS": 5,
# 	"LR_DECAY": True,
# }

# # GraphSage
# hp = {
# 	"NEG_SAMPLING": True,
# 	"NEG_SAMPLER": "unconsumed",  # random | unconsumed | popular
# 	"NUM_NEG": 5,
# 	"BATCH_SIZE": 1024,
# 	"LR": 1e-3,
# 	"LOSS_TYPE": "softmax",
# 	"TASK": "ranking",
# 	"EMBED_SIZE": 16,
# 	"LAYER_SIZES": (64, 64),
# 	"N_EPOCHS": 5,
# 	"LR_DECAY": True,
# 	"RECENT_NUM": 15,
# 	"RANDOM_NUM": 15
# }




print(hp)

{'NEG_SAMPLING': True, 'NEG_SAMPLER': 'unconsumed', 'NUM_NEG': 3, 'BATCH_SIZE': 4096, 'LR': 0.001, 'LOSS_TYPE': 'ranknet', 'TASK': 'ranking', 'EMBED_SIZE': 8, 'N_EPOCHS': 3, 'LR_DECAY': True, 'RECENT_NUM': 31, 'RANDOM_NUM': 31, 'HIDDEN_UNITS': (256, 128, 64)}


In [ ]:
def reset_state(name):
	tf.compat.v1.reset_default_graph()
	print("\n", "=" * 30, name, "=" * 30)


metrics = [
	"loss",
	"balanced_accuracy",
	"roc_auc",
	"pr_auc",
	"precision",
	"recall",
	"map",
	"ndcg",
]

# Convert to dict (libreco expects a dict)
config_dict = {
	'gpu_options': {
		'allow_growth': True,
		'per_process_gpu_memory_fraction': 0.95  # Reduce to 60% to leave room for the OS
	}
}

from recora.algorithms import (
	YouTubeRetrieval,
	YouTubeRanking,
	UserCF,
	FM,
	AutoInt,
	WideDeep,
	ItemCF,
	RNN4Rec,
	BPR,
	LightGCN,
	NGCF,
	Transformer,
	DeepFM,
	GraphSage
)

# reset_state("Transformer")
# model = Transformer(
# 	task=hp["TASK"],
# 	data_info=data_info,
# 	loss_type=hp["LOSS_TYPE"],
# 	embed_size=hp["EMBED_SIZE"],
# 	n_epochs=hp["N_EPOCHS"],
# 	lr=hp["LR"],
# 	lr_decay=hp["LR_DECAY"],
# 	reg=None,
# 	batch_size=hp["BATCH_SIZE"],
# 	use_bn=False,
# 	dropout_rate=None,
# 	num_heads=hp["NUM_HEAD"],
# 	num_tfm_layers=hp["NUM_TFM_LAYERS"],
# 	use_causal_mask=False,
# 	feat_agg_mode='concat',
# 	recent_num=hp["RECENT_NUM"],
# 	tf_sess_config=config_dict,
# )

# reset_state("BPR")
# model = BPR(
# 	task=hp["TASK"],
# 	data_info=data_info,
# 	loss_type=hp["LOSS_TYPE"],
# 	embed_size=hp["EMBED_SIZE"],
# 	n_epochs=hp["N_EPOCHS"],
# 	lr=hp["LR"],
# 	lr_decay=hp["LR_DECAY"],
# 	batch_size=hp["BATCH_SIZE"],
# 	num_neg=hp["NUM_NEG"],
# )

# reset_state("RNN4Rec")
# model = RNN4Rec(
# 	"ranking",
# 	data_info,
# 	rnn_type="gru",
# 	loss_type=hp["LOSS_TYPE"],
# 	embed_size=hp["EMBED_SIZE"],
# 	n_epochs=hp["N_EPOCHS"],
# 	lr=hp["LR"],
# 	lr_decay=hp["LR_DECAY"],
# 	hidden_units=hp["HIDDEN_UNITS"],
# 	reg=None,
# 	batch_size=hp["BATCH_SIZE"],
# 	num_neg=hp["NUM_NEG"],
# 	dropout_rate=None,
# 	recent_num=hp["RECENT_NUM"],
# 	tf_sess_config=config_dict,
# )

# reset_state("FM")
# model = FM(
# 	"ranking",
# 	data_info,
# 	loss_type="cross_entropy",
# 	embed_size=32,
# 	n_epochs=5,
# 	lr=1e-4,
# 	epsilon=1e-8,
# 	lr_decay=False,
# 	reg=1e-6,
# 	batch_size=BATCH_SIZE,
# 	num_neg=NUM_NEG,
# 	# num_neg=1,
# 	use_bn=False,
# 	dropout_rate=0.2,
# 	tf_sess_config=config_dict,
# 	# lower_upper_bound=(1,55)
# )

# reset_state("DeepFM")
# model = DeepFM(
# 	hp["TASK"],
# 	data_info,
# 	loss_type=hp["LOSS_TYPE"],
# 	embed_size=hp["EMBED_SIZE"],
# 	n_epochs=hp["N_EPOCHS"],
# 	lr=hp["LR"],
# 	lr_decay=hp["LR_DECAY"],
# 	reg=None,
# 	batch_size=hp["BATCH_SIZE"],
# 	num_neg=hp["NUM_NEG"],
# 	use_bn=False,
# 	dropout_rate=None,
# 	hidden_units=hp["HIDDEN_UNITS"],
# 	tf_sess_config=config_dict,
# )

# reset_state("WideDeep")
# model = WideDeep(
# 	"rating",
# 	data_info,
# 	loss_type="cross_entropy",
# 	embed_size=8,
# 	n_epochs=5,
# 	lr={"wide": 0.01, "deep": 1e-4},
# 	lr_decay=False,
# 	reg=None,
# 	batch_size=BATCH_SIZE,
# 	num_neg=1,
# 	use_bn=False,
# 	dropout_rate=None,
# 	hidden_units=(128, 64, 32),
# 	tf_sess_config=config_dict,
# 	lower_upper_bound=(1,5)
# )
# reset_state("AutoInt")
# model = AutoInt(
# 	"ranking",
# 	data_info,
# 	loss_type='bpr',
# 	embed_size=32,
# 	n_epochs=2,
# 	att_embed_size=(8, 8, 8),
# 	num_heads=2,
# 	use_residual=False,
# 	lr=1e-3,
# 	lr_decay=False,
# 	reg=None,
# 	batch_size=BATCH_SIZE,
# 	num_neg=1,
# 	use_bn=False,
# 	dropout_rate=None,
# 	tf_sess_config=config_dict,
# )

# reset_state("YoutubeRetrieval")
# model = YouTubeRetrieval(
# 	"ranking",
# 	data_info,
# 	loss_type="sampled_softmax",
# 	embed_size=32,
# 	norm_embed=False,
# 	n_epochs=10,
# 	lr=1.25*1e-3,
# 	lr_decay=True,
# 	reg=1e-6,
# 	batch_size=BATCH_SIZE,
# 	use_bn=False,
# 	dropout_rate=0.2,
# 	hidden_units=(256, 128, 64),
# 	tf_sess_config=config_dict,
# 	num_sampled_per_batch=BATCH_SIZE * NUM_NEG,
# 	sampler='log_uniform_candidate_sampler',
# 	recent_num=31,
# 	random_num=31
# 	# multi_sparse_combiner='normal'
# )
# #
reset_state("YouTubeRanking")
model = YouTubeRanking(
	hp["TASK"],
	data_info,
	loss_type=hp["LOSS_TYPE"],
	embed_size=hp["EMBED_SIZE"],
	n_epochs=hp["N_EPOCHS"],
	lr=hp["LR"],
	lr_decay=hp["LR_DECAY"],
	reg=1e-5,
	batch_size=hp["BATCH_SIZE"],
	num_neg=hp["NUM_NEG"],
	sampler=hp["NEG_SAMPLER"],
	use_bn=False,
	dropout_rate=0.1,
	hidden_units=hp["HIDDEN_UNITS"],
	tf_sess_config=config_dict,
	recent_num=hp["RECENT_NUM"],
	random_num=hp["RANDOM_NUM"]
)
#
# reset_state("LightGCN")
# model = LightGCN(
# 	hp["TASK"],
# 	data_info,
# 	loss_type=hp["LOSS_TYPE"],
# 	embed_size=hp["EMBED_SIZE"],
# 	n_layers=hp["N_LAYERS"],
# 	n_epochs=hp["N_EPOCHS"],
# 	lr=hp["LR"],
# 	batch_size=hp["BATCH_SIZE"],
# 	sampler=hp["NEG_SAMPLER"],
# 	num_neg=hp["NUM_NEG"],
# 	tf_sess_config=config_dict,
# )

# reset_state("NGCF")
# model = NGCF(
#     hp["TASK"],
#     data_info,
#     loss_type=hp["LOSS_TYPE"],
#     embed_size=hp["EMBED_SIZE"],
#     layer_sizes=hp["LAYER_SIZES"],
#     node_dropout_rate=hp["NODE_DROPOUT"],
#     message_dropout_rate=hp["MESSAGE_DROPOUT"],
#     n_epochs=hp["N_EPOCHS"],
#     lr=hp["LR"],
#     batch_size=hp["BATCH_SIZE"],
#     num_neg=hp["NUM_NEG"],
# )

# reset_state("GraphSage")
# model = GraphSage(
# 	hp["TASK"],
# 	data_info,
# 	loss_type=hp["LOSS_TYPE"],
# 	embed_size=hp["EMBED_SIZE"],
# 	layer_sizes=hp["LAYER_SIZES"],
# 	n_epochs=hp["N_EPOCHS"],
# 	lr=hp["LR"],
# 	lr_decay=hp["LR_DECAY"],
# 	batch_size=hp["BATCH_SIZE"],
# 	num_neg=hp["NUM_NEG"],
# 	sampler=hp["NEG_SAMPLER"],
#
# 	recent_num=hp["RECENT_NUM"],
# 	random_num=hp["RANDOM_NUM"],
# 	use_bn=False,
# 	dropout_rate=None,
# 	reg=None,
# 	tf_sess_config=config_dict,
# )

print("Hyperparameters: ")
print("-" * 40)
print(hp)
print("-" * 40)

model.fit(
	train_set,
	neg_sampling=hp["NEG_SAMPLING"],
	verbose=2,
	shuffle=True,
	metrics=metrics,
	eval_data=eval_set,
	eval_user_num=2500,
	k=TOP_K,
	eval_batch_size=hp["BATCH_SIZE"],
)


 ============================== YouTubeRanking ==============================
Hyperparameters: 
----------------------------------------
{'NEG_SAMPLING': True, 'NEG_SAMPLER': 'unconsumed', 'NUM_NEG': 3, 'BATCH_SIZE': 4096, 'LR': 0.001, 'LOSS_TYPE': 'ranknet', 'TASK': 'ranking', 'EMBED_SIZE': 8, 'N_EPOCHS': 3, 'LR_DECAY': True, 'RECENT_NUM': 31, 'RANDOM_NUM': 31, 'HIDDEN_UNITS': (256, 128, 64)}
----------------------------------------
Training start time: 2026-04-07 08:38:01
total params: 3,270,841 | embedding params: 3,145,849 | network params: 124,992
With lr_decay, epoch 1 learning rate: 0.0010000000474974513


train: 100%|██████████| 2983/2983 [03:44<00:00, 13.27it/s]


Epoch 1 elapsed: 224.836s
	 train_loss: 0.3539


eval_listwise: 100%|██████████| 41/41 [00:01<00:00, 29.01it/s]


	 eval log_loss: 0.6302
	 eval balanced_accuracy: 0.7257
	 eval roc_auc: 0.7967
	 eval pr_auc: 0.4734
	 eval precision@20: 0.0552
	 eval recall@20: 0.4029
	 eval map@20: 0.2115
	 eval ndcg@20: 0.3138
With lr_decay, epoch 2 learning rate: 0.0009600000339560211


...

eval_listwise: 100%|██████████| 41/41 [00:01<00:00, 29.38it/s]


	 eval log_loss: 0.6431
	 eval balanced_accuracy: 0.7204
	 eval roc_auc: 0.7873
	 eval pr_auc: 0.4494
	 eval precision@20: 0.0537
	 eval recall@20: 0.3895
	 eval map@20: 0.1984
	 eval ndcg@20: 0.2996
With lr_decay, epoch 3 learning rate: 0.0009215999743901193


train: 100%|██████████| 2983/2983 [03:54<00:00, 12.75it/s]


Epoch 3 elapsed: 234.044s
	 train_loss: 0.2369


eval_pointwise:  30%|███       | 420/1385 [01:14<05:18,  3.03it/s]  

In [67]:
# 20.6
from recora.evaluation import evaluate, print_metrics

eval_result = evaluate(
	model=model,
	data=eval_set,
	neg_sampling=True,
	eval_batch_size=hp["BATCH_SIZE"],
	k=5,
	metrics=metrics,
	sample_user_num=5000,
)

NameError: name 'BATCH_SIZE' is not defined

In [35]:
eval_result

{'loss': 0.422380005055822,
 'balanced_accuracy': 0.8192213159610361,
 'roc_auc': 0.900754192002259,
 'pr_auc': 0.8984094544084269,
 'precision': 0.04172000000000001,
 'recall': 0.04303334687485299,
 'map': 0.09695166720449924,
 'ndcg': 0.1199585077721801}

In [36]:
# model = YouTubeRetrieval(
# 	"ranking",
# 	data_info,
# 	loss_type="sampled_softmax",
# 	embed_size=32,
# 	norm_embed=False,
# 	n_epochs=10,
# 	lr=1.25*1e-3,
# 	lr_decay=True,
# 	reg=1e-6,
# 	batch_size=512,
# 	use_bn=False,
# 	dropout_rate=0.2,
# 	hidden_units=(256, 128, 64),
# 	tf_sess_config=config_dict,
# 	num_sampled_per_batch=512 * 15,
# 	recent_num=31,
# 	random_num=31
# 	# multi_sparse_combiner='normal'
# )

# Epoch 10 elapsed: 178.519s
# 	 train_loss: 5.8241
# eval_pointwise: 100%|██████████| 3788/3788 [00:01<00:00, 3569.97it/s]
# eval_listwise: 100%|██████████| 5000/5000 [00:05<00:00, 950.57it/s]
# 	 eval log_loss: 0.2397
# 	 eval balanced_accuracy: 0.9053
# 	 eval roc_auc: 0.9718
# 	 eval pr_auc: 0.9684
# 	 eval precision@20: 0.0295
# 	 eval recall@20: 0.2478
# 	 eval map@20: 0.1391
# 	 eval ndcg@20: 0.2134

In [ ]:
# model = YouTubeRanking(
# 	"ranking",
# 	data_info,
# 	loss_type="ranknet",
# 	embed_size=32,
# 	n_epochs=5,
# 	lr=2e-3,
# 	lr_decay=False,
# 	reg=1e-6,
# 	batch_size=BATCH_SIZE,
# 	num_neg=NUM_NEG,
# 	sampler=NEG_SAMPLER,
# 	use_bn=False,
# 	dropout_rate=0.2,
# 	hidden_units=(256, 128, 64),
# 	tf_sess_config=config_dict,
# 	recent_num=10,
# )
#
# 	 # eval log_loss: 0.3646
# 	 # eval balanced_accuracy: 0.8449
# 	 # eval roc_auc: 0.9204
# 	 # eval pr_auc: 0.7349
# 	 # eval precision@20: 0.0388
# 	 # eval recall@20: 0.1612
# 	 # eval map@20: 0.1497
# 	 # eval ndcg@20: 0.2412


In [37]:
# from libreco.data import DataInfo

# SAVE_PATH = 'youtube-retr'
# SAVE_MODEL_NAME = 'youtuberetrieval'
# # save data_info, specify model save folder
# data_info.save(
# 	path=SAVE_PATH,
# 	model_name=SAVE_MODEL_NAME
# )
# # set manual=True will use `numpy` to save model
# # set manual=False will use `tf.train.Saver` to save model
# # set inference=True will only save the necessary variables for prediction and recommendation
# model.save(
# 	path=SAVE_PATH,
# 	model_name=SAVE_MODEL_NAME,
# 	manual=False,
# 	inference_only=True
# )

In [39]:
result = model.recommend_user(
	"30953879",
	n_rec=1000,
	filter_consumed=False,
	# user_feats={
	# 	"order_hour": 15 / 23
	# }
)

result = pd.DataFrame(result)
result[['brand_id', 'category_id']] = (
	result["30953879"]
	.str.split("012", expand=True, n=1)  # n=1 ensures we only split once
)
# Optional: convert to integer if they are numeric IDs
result['brand_id'] = pd.to_numeric(result['brand_id'], errors='coerce')
result['category_id'] = pd.to_numeric(result['category_id'], errors='coerce')
brand_category_titles = pd.read_csv('./sample_data/brand_category_titles.csv')
result = result.merge(brand_category_titles, on=['brand_id', 'category_id'], how='left')
result[result['category_id'] == 69].head(20)

,30953879,brand_id,category_id,category_title,brand_title
8,33701269,337,69.0,شیر,دامداران
20,33301269,333,69.0,شیر,مزرعه ماهشام
27,33601269,336,69.0,شیر,میهن
31,36801269,368,69.0,شیر,پگاه
54,24001269,240,69.0,شیر,کاله
61,34701269,347,69.0,شیر,چوپان
94,33501269,335,69.0,شیر,پاک
99,34301269,343,69.0,شیر,پاک
101,20701269,207,69.0,شیر,متفرقه
125,38501269,385,69.0,شیر,روزانه


In [256]:
result = train_data.loc[train_data['user'] == "30953879"].sort_values(by='label', ascending=False).reset_index()
result[['brand_id', 'category_id']] = (
	result['item']
	.str.split("012", expand=True, n=1)  # n=1 ensures we only split once
)
result['brand_id'] = pd.to_numeric(result['brand_id'], errors='coerce')
result['category_id'] = pd.to_numeric(result['category_id'], errors='coerce')
brand_category_titles = pd.read_csv('./sample_data/brand_category_titles.csv')
result = result.merge(brand_category_titles, on=['brand_id', 'category_id'], how='left')
result.loc[result['category_id'] == 651, ['user', 'item', 'label', 'category_title', 'brand_title']]

,user,item,label,category_title,brand_title
0,30953879,12627012651,55.0,سیگار,کاوالو
1,30953879,7311012651,45.0,سیگار,وینستون
2,30953879,16843012651,21.0,سیگار,کالوالو
4,30953879,11653012651,3.0,سیگار,بیستون
9,30953879,14051012651,1.0,سیگار,اسی
16,30953879,1447012651,1.0,سیگار,بهمن
18,30953879,12257012651,1.0,سیگار,کمل


In [ ]:
import tensorflow as tf
from libreco.data import DataInfo
from libreco.algorithms import TwoTower, YouTubeRetrieval

# important to reset graph if model is loaded in the same shell.
tf.compat.v1.reset_default_graph()
# load data_info
data_info = DataInfo.load(
	SAVE_PATH, model_name=SAVE_MODEL_NAME
)
print(data_info)
# load model, should specify the model name, e.g., DeepFM
model: YouTubeRetrieval = YouTubeRetrieval.load(
	path=SAVE_PATH, model_name=SAVE_MODEL_NAME, data_info=data_info, manual=True
)

In [ ]:
# model.user_embeds_np[data_info.user2id[30953879]].shape
model.user_embeds_np.shape

In [ ]:
import faiss
import numpy as np
import pandas as pd

faiss_index: faiss.swigfaiss.IndexIVFFlat = faiss.read_index('./embed_model/faiss_index.bin')
user_embedding = model.get_user_embedding(30953879, include_bias=True).astype(np.float32).reshape(1, -1)
distances, ids = faiss_index.search(user_embedding, 100)

In [ ]:
distances

In [ ]:
result = pd.DataFrame(list(map(lambda item: list(map(lambda x: int(x), item.split("012"))), [data_info.id2item[x] for x in ids.ravel().tolist()])),
                      columns=['brand_id', 'category_id'])
result['distances'] = distances.ravel()

brand_category_titles = pd.read_csv('brand_category_titles.csv')
result = result.merge(brand_category_titles, on=['brand_id', 'category_id'], how='left')

In [ ]:
# result[result['category_id'] == 69]
result

# Init `embed_deploy`

In [ ]:
from libserving.serialization import embed2redis, save_embed, save_faiss_index

path = "embed_model"  # specify model saving directory
save_embed(path, model)  # save model in json format

In [ ]:
from libserving.serialization.redis import redis_connection, model_name2redis, id_mapping2redis, user_consumed2redis, user_embed2redis
import ujson
import os

host = "localhost"
port = 6379
db = 0
chunk_size = 10000
with redis_connection(host, port, db) as r:
	model_name2redis(path, r)
	print("\nmodel_name2redis ...")

with redis_connection(host, port, db) as r:
	id_mapping2redis(path, r)
	print("\nid_mapping2redis ...")

with redis_connection(host, port, db) as r:
	with open(os.path.join(path, "user_consumed.json")) as f:
		data = ujson.load(f)

	pipe = r.pipeline()
	for i, (u, items) in enumerate(data.items()):
		pipe.hset("user_consumed", u, ujson.dumps(items))
		if (i + 1) % chunk_size == 0:
			pipe.execute()
			pipe = r.pipeline()
	pipe.execute()
	print("\nuser_consumed2redis ...")

with redis_connection(host, port, db) as r:
	with open(os.path.join(path, "user_embed.json")) as f:
		user_embeds = ujson.load(f)

	pipe = r.pipeline()
	for i, (u, embed) in enumerate(user_embeds.items()):
		pipe.hset("user_embed", u, ujson.dumps(embed))
		if (i + 1) % chunk_size == 0:
			pipe.execute()
			pipe = r.pipeline()
	pipe.execute()
	print("\nuser_embed2redis ...")

In [ ]:
from libserving.serialization import save_faiss_index

print(path, model)
save_faiss_index(path, model)

In [ ]:
import requests
import json

payload = {
	"user": "30953879",
	"n_rec": 100
}

result = requests.post(
	"http://127.0.0.1:8000/embed/recommend",
	headers={"Content-Type": "application/json"},
	data=json.dumps(payload)
).json()

In [ ]:
result[['brand_id', 'category_id']] = (
	result['item']
	.str.split("012", expand=True, n=1)  # n=1 ensures we only split once
)

# Optional: convert to integer if they are numeric IDs
result['brand_id'] = pd.to_numeric(result['brand_id'], errors='coerce')
result['category_id'] = pd.to_numeric(result['category_id'], errors='coerce')

# Init `online_deploy`

In [ ]:
from libserving.serialization import save_online, online2redis

path = "online_model"  # specify model saving directory
save_online(path, model, version=1)  # save model in json format

In [ ]:
from libserving.serialization.redis import redis_connection, model_name2redis, id_mapping2redis
import ujson
import os

host = "localhost"
port = 6379
db = 0
chunk_size = 10000  # ← same as you used before

# ------------------------------------------------------------------
# 1. model_name + id_mapping (small, unchanged)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	model_name2redis(path, r)
	print("\nmodel_name2redis ...")

with redis_connection(host, port, db) as r:
	id_mapping2redis(path, r)
	print("\nid_mapping2redis ...")

# ------------------------------------------------------------------
# 2. user_consumed (your existing chunked version)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	with open(os.path.join(path, "user_consumed.json")) as f:
		data = ujson.load(f)

	pipe = r.pipeline()
	for i, (u, items) in enumerate(data.items()):
		pipe.hset("user_consumed", u, ujson.dumps(items))
		if (i + 1) % chunk_size == 0:
			pipe.execute()
			pipe = r.pipeline()
	pipe.execute()
	print("\nuser_consumed2redis ...")

# ------------------------------------------------------------------
# 3. features2redis → FULLY CHUNKED (the heavy part for online models)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	feature_path = os.path.join(path, "features.json")
	with open(feature_path) as f:
		feats = ujson.load(f)

	r.set("n_users", feats["n_users"])
	r.set("n_items", feats["n_items"])
	if "max_seq_len" in feats:
		r.set("max_seq_len", feats["max_seq_len"])

	# user_sparse_values (per-user)
	if "user_sparse_col_index" in feats:
		r.hset("feature", "user_sparse", 1)
		r.set("user_sparse_col_index", ujson.dumps(feats["user_sparse_col_index"]))
		pipe = r.pipeline()
		for i, vals in enumerate(feats["user_sparse_values"]):
			pipe.hset("user_sparse_values", str(i), ujson.dumps(vals))
			if (i + 1) % chunk_size == 0:
				pipe.execute()
				pipe = r.pipeline()
		pipe.execute()

	# item_sparse_values (list push)
	if "item_sparse_col_index" in feats:
		r.hset("feature", "item_sparse", 1)
		r.set("item_sparse_col_index", ujson.dumps(feats["item_sparse_col_index"]))
		pipe = r.pipeline()
		for i, vals in enumerate(feats["item_sparse_values"]):
			pipe.rpush("item_sparse_values", ujson.dumps(vals))
			if (i + 1) % chunk_size == 0:
				pipe.execute()
				pipe = r.pipeline()
		pipe.execute()

	# user_dense_values (per-user)
	if "user_dense_col_index" in feats:
		r.hset("feature", "user_dense", 1)
		r.set("user_dense_col_index", ujson.dumps(feats["user_dense_col_index"]))
		pipe = r.pipeline()
		for i, vals in enumerate(feats["user_dense_values"]):
			pipe.hset("user_dense_values", str(i), ujson.dumps(vals))
			if (i + 1) % chunk_size == 0:
				pipe.execute()
				pipe = r.pipeline()
		pipe.execute()

	# item_dense_values (list push)
	if "item_dense_col_index" in feats:
		r.hset("feature", "item_dense", 1)
		r.set("item_dense_col_index", ujson.dumps(feats["item_dense_col_index"]))
		pipe = r.pipeline()
		for i, vals in enumerate(feats["item_dense_values"]):
			pipe.rpush("item_dense_values", ujson.dumps(vals))
			if (i + 1) % chunk_size == 0:
				pipe.execute()
				pipe = r.pipeline()
		pipe.execute()

	print("\nfeatures2redis ...")

# ------------------------------------------------------------------
# 4. user_sparse2redis (small, no chunk needed)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	user_sparse_fields_path = os.path.join(path, "user_sparse_fields.json")
	if os.path.exists(user_sparse_fields_path):
		with open(user_sparse_fields_path) as f:
			user_sparse_fields = ujson.load(f)
		r.hset("user_sparse_fields", mapping=user_sparse_fields)

	user_sparse_idx_mapping_path = os.path.join(path, "user_sparse_idx_mapping.json")
	if os.path.exists(user_sparse_idx_mapping_path):
		with open(user_sparse_idx_mapping_path) as f:
			user_sparse_idx_mapping = ujson.load(f)
		for col, idx_mapping in user_sparse_idx_mapping.items():
			col_name = f"user_sparse_idx_mapping__{col}"
			r.hset(col_name, mapping=idx_mapping)

	print("\nuser_sparse2redis ...")

# ------------------------------------------------------------------
# 5. user_dense2redis (small, no chunk needed)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	user_dense_fields_path = os.path.join(path, "user_dense_fields.json")
	if os.path.exists(user_dense_fields_path):
		with open(user_dense_fields_path) as f:
			user_dense_fields = ujson.load(f)
		r.hset("user_dense_fields", mapping=user_dense_fields)

	print("\nuser_dense2redis ...")

print("\n✅ All online2redis data loaded into Redis with chunking!")

In [ ]:
from libserving.serialization import save_faiss_index

save_faiss_index(path, model)

In [ ]:
result = model.recommend_user(
	30953879,
	n_rec=10,
	filter_consumed=False,
	# user_feats={
	# 	"order_hour": 20
	# }
)

result = pd.DataFrame(result)
result[['brand_id', 'category_id']] = (
	result[30953879]
	.str.split("012", expand=True, n=1)  # n=1 ensures we only split once
)
# Optional: convert to integer if they are numeric IDs
result['brand_id'] = pd.to_numeric(result['brand_id'], errors='coerce')
result['category_id'] = pd.to_numeric(result['category_id'], errors='coerce')
brand_category_titles = pd.read_csv('brand_category_titles.csv')
result = result.merge(brand_category_titles, on=['brand_id', 'category_id'], how='left')
result

In [ ]:
json.dumps(payload)

In [ ]:
import requests
import json

payload = {
	"user": 30953879,
	"n_rec": 10,
	"filter_consumed": False,
	"return_scores": False,
	"user_feats": {
		"order_hour": 12
	}
}

result = requests.post(
	"http://127.0.0.1:8000/online/recommend",
	headers={"Content-Type": "application/json"},
	data=json.dumps(payload)
).json()

result = pd.DataFrame(result)
result[['brand_id', 'category_id']] = (
	result['recommendations']
	.str.split("012", expand=True, n=1)  # n=1 ensures we only split once
)

# Optional: convert to integer if they are numeric IDs
result['brand_id'] = pd.to_numeric(result['brand_id'], errors='coerce')
result['category_id'] = pd.to_numeric(result['category_id'], errors='coerce')
brand_category_titles = pd.read_csv('brand_category_titles.csv')
result = result.merge(brand_category_titles, on=['brand_id', 'category_id'], how='left')
result

### Read `brand_category_titles` data

In [ ]:
BRAND_CATEGORY_SQL = """
SELECT category_level2_id                                        AS category_id,
       brand_id                                                  AS brand_id,
       dictGet('KF.dim_categories', 'title', category_level2_id) AS category_title,
       dictGet('KF.dim_brands', 'title', brand_id)               AS brand_title
FROM KF.plane_products
GROUP BY category_level2_id, brand_id
"""

brand_category_titles = client.query_df(BRAND_CATEGORY_SQL)

In [ ]:
brand_category_titles.to_csv('brand_category_titles.csv', index=False)

In [ ]:
brand_category_titles = pd.read_csv('brand_category_titles.csv')